In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

import sys
import os

sys.path.append('..')
os.chdir('..')  

from src.utils.config_loader import load_config
from src.data_pipeline.preprocess import *
from src.data_pipeline.features import *
config = load_config("configs/data_config.yaml")
print("Config loaded successfully ✅")

https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/review_categories/Electronics.jsonl.gz
Config loaded successfully ✅


In [2]:
# Load ratings data
raw_path = config['paths']['raw_data']

df_ratings = pd.read_csv(raw_path + "Electronics_ratings.csv")

print(f"Shape: {df_ratings.shape}")
print(f"\nColumns: {df_ratings.columns.tolist()}")
print(f"\nFirst 5 rows:")
df_ratings.head()

Shape: (100000, 10)

Columns: ['rating', 'title', 'text', 'images', 'asin', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote', 'verified_purchase']

First 5 rows:


,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,3.0,Smells like gasoline! Going back!,First & most offensive: they reek of gasoline ...,[{'small_image_url': 'https://m.media-amazon.c...,B083NRGZMM,B083NRGZMM,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1658185117948,0,True
1,1.0,Didn’t work at all lenses loose/broken.,These didn’t work. Idk if they were damaged in...,[],B07N69T6TM,B07N69T6TM,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1592678549731,0,True
2,5.0,Excellent!,I love these. They even come with a carry case...,[],B01G8JO5F2,B01G8JO5F2,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1523093017534,0,True
3,5.0,Great laptop backpack!,I was searching for a sturdy backpack for scho...,[],B001OC5JKY,B001OC5JKY,AGGZ357AO26RQZVRLGU4D4N52DZQ,1290278495000,18,True
4,5.0,Best Headphones in the Fifties price range!,I've bought these headphones three times becau...,[],B013J7WUGC,B07CJYMRWM,AG2L7H23R5LLKDKLBEF2Q3L2MVDA,1676601581238,0,True


In [3]:
prep_config = config['preprocessing']

df_ratings = drop_useless_columns(df_ratings, prep_config['drop_columns'])
df_ratings = remove_missing_values(df_ratings, prep_config['missing_values']['subset'])
df_ratings = remove_duplicates(df_ratings)
df_ratings = convert_timestamp(df_ratings)
df_ratings = convert_to_integer(df_ratings, prep_config['convert_to_integer']['columns'])
df_ratings = handle_outliers(df_ratings, prep_config['outliers']['columns'], prep_config['outliers']['method'])
df_ratings = detect_spam(df_ratings, prep_config['spam_detection']['max_reviews_per_day'], prep_config['spam_detection']['min_time_gap'])
df_ratings = add_review_weight(df_ratings)
df_ratings = filter_text(df_ratings,column='text',
                         min_words = prep_config['text_filter']['min_words'],
                         max_words = prep_config['text_filter']['max_words'])
df_ratings, encoders = encode_labels(df_ratings, prep_config['encode']['columns'])
train_df, test_df = time_based_split(df_ratings, "timestamp", prep_config['split']['test_year'])

[INFO] Dropped columns: ['images']
[INFO] Removed 18 rows with missing values
[INFO] Removed 12 duplicate rows
[INFO] Converted 'timestamp' to datetime
[INFO] Converted 'verified_purchase' to integer
[INFO] Capped 8789 outliers in 'helpful_vote' using method='upper'
[INFO] Removed 528 spam users
[INFO] Added weight column: verified=1.0, unverified=0.7
[INFO] Valid texts for NLP: 72483
[INFO] Marked 7010 short texts as None
[INFO] Truncated 3833 long texts to 250 words
[INFO] Encoded 'user_id' — 14789 unique labels
[INFO] Encoded 'asin' — 54384 unique labels
[INFO] Train: 67067 rows (84.4%)
[INFO] Test:  12426 rows (15.6%)


In [4]:
train_df = add_user_segment(train_df)
train_df = add_features(train_df)
train_df, scaler = normalize(train_df, 'helpful_vote')

test_df, _ = normalize(test_df, 'helpful_vote', scaler=scaler)

[INFO] User segments:
user_segment
Heavy     30569
Medium    26869
Light      9629
Name: count, dtype: int64
[INFO] Added features: user_verified_ratio, item_avg_rating, is_weekend
[INFO] Fitted and normalized 'helpful_vote'
[INFO] Transformed 'helpful_vote' using existing scaler


In [5]:
# Build User-Item Matrix
matrix, pivot_df = build_user_item_matrix(train_df)

[INFO] Matrix shape: (13668, 45381)
[INFO] Sparsity: 99.9892%


In [6]:
# Save processed data
processed_path = config['paths']['processed_data']

train_df.to_parquet(processed_path + 'train.parquet', index=False)
test_df.to_parquet(processed_path + 'test.parquet', index=False)

print(f"[INFO] Train saved: {train_df.shape}")
print(f"[INFO] Test saved:  {test_df.shape}")

[INFO] Train saved: (67067, 14)
[INFO] Test saved:  (12426, 10)


In [7]:
import joblib

joblib.dump(encoders, processed_path + 'encoders.pkl')
joblib.dump(scaler, processed_path + 'scaler.pkl')

print("[INFO] Encoders and scaler saved ✅")

[INFO] Encoders and scaler saved ✅


In [ ]:
import joblib

joblib.dump(matrix, processed_path + 'user_item_matrix.pkl')
joblib.dump(pivot_df, processed_path + 'pivot_df.pkl')

['data/processed/pivot_df.pkl']